In [10]:
import random
import copy
from agents import Agent, Environment, Food, Water, Bump, Direction

class SmartBlindDog(Agent):
    def __init__(self, program=None):
        super().__init__(program)
        self.location = [0, 0]
        self.direction = Direction(Direction.D)
        self.visited = set()
        self.visited.add(tuple(self.location))

    def moveforward(self, success=True):
        if not success:
            return
        if self.direction.direction == Direction.R:
            self.location[0] += 1
        elif self.direction.direction == Direction.L:
            self.location[0] -= 1
        elif self.direction.direction == Direction.D:
            self.location[1] += 1
        elif self.direction.direction == Direction.U:
            self.location[1] -= 1
        
        self.visited.add(tuple(self.location))

    def turn(self, d):
        self.direction = self.direction + d

    def get_ahead_location(self):
        x, y = self.location[0], self.location[1]
        if self.direction.direction == Direction.R:
            return [x + 1, y]
        elif self.direction.direction == Direction.L:
            return [x - 1, y]
        elif self.direction.direction == Direction.D:
            return [x, y + 1]
        elif self.direction.direction == Direction.U:
            return [x, y - 1]
        return [x, y]


class Park2D(Environment):
    def __init__(self, width=5, height=5):
        super().__init__()
        self.width = width
        self.height = height

    def is_inbounds(self, location):
        return 0 <= location[0] < self.width and 0 <= location[1] < self.height

    def percept(self, agent):
        things = self.list_things_at(agent.location)
        ahead = agent.get_ahead_location()
        if not self.is_inbounds(ahead):
            things.append(Bump())
        return things

    def execute_action(self, agent, action):
        if action == 'eat':
            items = self.list_things_at(agent.location, tclass=Food)
            if len(items) != 0:
                print(f"🐶 Dog: Eat Food at {agent.location}")
                self.delete_thing(items[0])
        elif action == 'drink':
            items = self.list_things_at(agent.location, tclass=Water)
            if len(items) != 0:
                print(f"🐶 Dog: Drink Water at {agent.location}")
                self.delete_thing(items[0])
        elif action == 'moveforward':
            ahead = agent.get_ahead_location()
            if self.is_inbounds(ahead):
                agent.moveforward(True)
                print(f"🐶 Dog: Moved forward to {agent.location}")
            else:
                agent.moveforward(False)
                print(f"⚠️ Dog: BUMPED wall at {agent.location}!")
        elif action == 'turnright':
            agent.turn(Direction.R)
            print(f"🔄 Dog: Turned Right (Facing {agent.direction.direction})")
        elif action == 'turnleft':
            agent.turn(Direction.L)
            print(f"🔄 Dog: Turned Left (Facing {agent.direction.direction})")


def smart_program(percepts):
    for p in percepts:
        if isinstance(p, Food):
            return 'eat'
        elif isinstance(p, Water):
            return 'drink'

    has_bump = any(isinstance(p, Bump) for p in percepts)
    
    if has_bump:
        return random.choice(['turnright', 'turnleft'])

    return random.choice(['moveforward', 'moveforward', 'turnright', 'turnleft'])


if __name__ == "__main__":
    park = Park2D(5, 5)
    
    dog = SmartBlindDog(smart_program)
    park.add_thing(dog, [0, 0])

    park.add_thing(Food(), [0, 2])
    park.add_thing(Water(), [2, 2])
    park.add_thing(Food(), [4, 1])

    print("--- 🚀 เริ่มการจำลอง Smart BlindDog ---")
    print(f"ตำแหน่งเริ่มต้น: {dog.location}\n")

    park.run(15)

    print("\n--- 📊 สรุปผลการทำงาน ---")
    print(f"พิกัดทั้งหมดที่เคยเดินผ่าน (Memory): {dog.visited}")
    print(f"จำนวนพื้นที่ที่สำรวจได้: {len(dog.visited)} ช่อง")

--- 🚀 เริ่มการจำลอง Smart BlindDog ---
ตำแหน่งเริ่มต้น: [0, 0]

🐶 Dog: Moved forward to [0, 1]
🔄 Dog: Turned Left (Facing right)
🐶 Dog: Moved forward to [1, 1]
🔄 Dog: Turned Right (Facing down)
🔄 Dog: Turned Left (Facing right)
🔄 Dog: Turned Right (Facing down)
🐶 Dog: Moved forward to [1, 2]
🔄 Dog: Turned Right (Facing left)
🔄 Dog: Turned Right (Facing up)
🐶 Dog: Moved forward to [1, 1]
🐶 Dog: Moved forward to [1, 0]
🔄 Dog: Turned Right (Facing right)
🐶 Dog: Moved forward to [2, 0]
🔄 Dog: Turned Right (Facing down)
🔄 Dog: Turned Right (Facing left)

--- 📊 สรุปผลการทำงาน ---
พิกัดทั้งหมดที่เคยเดินผ่าน (Memory): {(0, 1), (1, 2), (0, 0), (1, 1), (2, 0), (1, 0)}
จำนวนพื้นที่ที่สำรวจได้: 6 ช่อง
